In [ ]:
# Exploring data extracted from curated_database.py

In [3]:
import csv, json

In [4]:
ncbi_accessions=[]
csv_path = "../uvsx/data/curated_database/ncbi_query_search_metadata.csv"
with open(csv_path, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        accession = row.get("accession")
        if accession:
            ncbi_accessions.append(accession)
print(f"NCBI Accessions: {len(ncbi_accessions)}")

#Get all query search accessions
uniprot_search_accessions=[]
with open("../uvsx/data/curated_database/uniprot_query_search_metadata.json") as f:
    data = json.load(f)
    for entry in data:
        accession = entry.get("primaryAccession")  # UniProt accession
        uniprot_search_accessions.append(accession)
print(f"Uniprot Accessions: {len(uniprot_search_accessions)}")

#Get all interpro match accessions
domain_match_accessions=[]
with open("../uvsx/data/curated_database/interpro_domain_matches_metadata.json") as f:
    data = json.load(f)
    for entry in data:
        accession = entry.get("primaryAccession")  # UniProt accession
        domain_match_accessions.append(accession)
print(f"Domain match Accessions: {len(domain_match_accessions)}")

NCBI Accessions: 17950
Uniprot Accessions: 7488
Domain match Accessions: 1048


26486

In [19]:
# Find the number of unique accessions
overlap=0
for accession in ncbi_accessions:
    if accession in uniprot_search_accessions:
        overlap+=1
        print(f"NCBI {accession} is duplicated in uniprot")
print(f'Overlap: {overlap}')

NCBI P32270 is duplicated in uniprot
NCBI P04529 is duplicated in uniprot
NCBI O21960 is duplicated in uniprot
NCBI O21959 is duplicated in uniprot
NCBI O21958 is duplicated in uniprot
NCBI O21957 is duplicated in uniprot
NCBI O21956 is duplicated in uniprot
NCBI O21955 is duplicated in uniprot
NCBI O21954 is duplicated in uniprot
NCBI O21953 is duplicated in uniprot
NCBI O21952 is duplicated in uniprot
NCBI O21951 is duplicated in uniprot
NCBI O21950 is duplicated in uniprot
NCBI O21949 is duplicated in uniprot
NCBI O21947 is duplicated in uniprot
NCBI O21948 is duplicated in uniprot
NCBI Q06728 is duplicated in uniprot
NCBI Q06727 is duplicated in uniprot
NCBI P04537 is duplicated in uniprot
NCBI P03690 is duplicated in uniprot
NCBI P41662 is duplicated in uniprot
NCBI P35926 is duplicated in uniprot
NCBI P04892 is duplicated in uniprot
NCBI P14819 is duplicated in uniprot
Overlap: 24


In [20]:
# Find the number of unique accessions
overlap=0
for accession in ncbi_accessions:
    if accession in domain_match_accessions:
        overlap+=1
        print(f"NCBI {accession} is duplicated in Domain match")
print(f'Overlap: {overlap}')

NCBI P04529 is duplicated in Domain match
Overlap: 1


In [21]:
# Find the number of unique accessions
overlap=0
for accession in domain_match_accessions:
    if accession in uniprot_search_accessions:
        overlap+=1
        print(f"Domain Match {accession} is duplicated in Uniprot")
print(f'Overlap: {overlap}')

Domain Match P04529 is duplicated in Uniprot
Domain Match A0A097J1I5 is duplicated in Uniprot
Domain Match A0A097J665 is duplicated in Uniprot
Domain Match A0A0K0QSG4 is duplicated in Uniprot
Domain Match A0A345AZ21 is duplicated in Uniprot
Domain Match A0A384SXS9 is duplicated in Uniprot
Domain Match A0A097J2C1 is duplicated in Uniprot
Domain Match A0A218MA15 is duplicated in Uniprot
Domain Match A0A384SHD3 is duplicated in Uniprot
Domain Match A0A449C4B8 is duplicated in Uniprot
Domain Match E3SFD3 is duplicated in Uniprot
Domain Match Q06ET6 is duplicated in Uniprot
Domain Match A0A249XY36 is duplicated in Uniprot
Domain Match A0A2Z5WL51 is duplicated in Uniprot
Domain Match A0A0A7HCT1 is duplicated in Uniprot
Domain Match A0A384T943 is duplicated in Uniprot
Domain Match A0A384TAA8 is duplicated in Uniprot
Domain Match I7KRM0 is duplicated in Uniprot
Domain Match S5M338 is duplicated in Uniprot
Domain Match C3V228 is duplicated in Uniprot
Domain Match A0A6M3YSR9 is duplicated in Uni

In [22]:
1048 + 1 + 24

1073

In [31]:
# Looking at metadata
with open('../uvsx/data/curated_database/interpro_domain_matches_metadata.json', 'r') as file:
    data = json.load(file)

print(len(data))

1048


In [5]:
unique_accessions=[]
for acc in uniprot_search_accessions:
    if acc not in unique_accessions:
        unique_accessions.append(acc)
for acc in ncbi_accessions:
    if acc not in unique_accessions:
        unique_accessions.append(acc)
for acc in domain_match_accessions:
    if acc not in unique_accessions:
        unique_accessions.append(acc)

In [309]:
len(unique_accessions)

25414

In [ ]:
#
#
#
# Above gets non dulicate accessions
#
#
#

In [146]:
import os
from collections import defaultdict

def read_fasta_sequences(filepath):
    sequences = {}
    seq = ""
    header = None

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if header and seq:
                    sequences[header] = seq

                header = line.split(' ')[0][1:]

                if '|' in header:
                    header = header.split('|')[1]
                if '.' in header:
                    header = header.split('.')[0]

                seq = ""
            else:
                seq += line

        if header and seq:
            sequences[header] = seq

    return sequences


def find_matching_sequences(file_list):
    seq_map = defaultdict(lambda: defaultdict(list))

    for file in file_list:
        file_name = os.path.split(file)[1]
        data = read_fasta_sequences(file)

        for header, seq in data.items():
            seq_map[seq][file_name].append(header)

    matches = []

    for seq, file_dict in seq_map.items():
        if len(file_dict) > 1:  # appears in more than one file
            matches.append({
                "sequence": seq,
                "files": file_dict
            })

    return matches

In [147]:
files=['../uvsx/data/curated_database/uniprot_query_search.fasta', '../uvsx/data/curated_database/ncbi_query_search.fasta']
matches = find_matching_sequences(files)

print(f"{len(matches)} matches found")

738 matches found


In [197]:
with open("../uvsx/data/curated_database/uniprot_query_search_metadata.json") as f:
    data = json.load(f)
lookup = {
    entry.get("primaryAccession"): entry.get("organism", {}).get("taxonId")
    for entry in data
}

tax_ids = []

for i in range(len(matches)):
    id_list = matches[i].get('files', {}).get('uniprot_query_search.fasta', [])

    for acc in id_list:
        tax = lookup.get(acc)  # O(1) lookup

        tax_ids.append({acc: tax})

In [202]:
from collections import defaultdict

grouped = defaultdict(list)

for d in tax_ids:
    for acc, tax in d.items():
        grouped[tax].append(acc)

print(dict(grouped))

#NOW I HAVE THE TAX IDS AND WHICH ACCESSIONS SHARE THEM. I WILL USE THIS WITH MATCHES TO SEE IF THERE ARE FOUND IN THE SAME LISTS. MEANING THEIR SEQUENCE IS THE SAME AND THEY HAVE THE SAME TAXON ID.

{10665: ['P04529', 'A0A023ZVM8', 'P32270', 'P04537', 'P03690', 'O36166'], 2681598: ['A0A7S9XGY1'], 697289: ['A0A097J734'], 697290: ['A0A097J7Q4'], 3234042: ['A0AB39C9X7'], 10666: ['Q06728', 'A0A346FJC7'], 2060721: ['Q06727', 'A0A2Z5WKD2'], 1651198: ['A0A384T943'], 1651199: ['A0A384TAA8'], 2681602: ['Q06ET6'], 10693: ['C3V228'], 2681597: ['C3V1A2'], 1351740: ['S5M338'], 1651202: ['A0A384SHD3'], 1112578: ['A0A097J1I5'], 134822: ['A0A097J665'], 69610: ['A0A097J2C1', 'O21951'], 2079316: ['A0A2K9VFU7'], 697291: ['A0A097J376'], 2079315: ['A0A2K9VF90'], 1837867: ['A0A173GAA3'], 69608: ['A0A097J4N0', 'O21954'], 69609: ['A0A097J5G0', 'O21957'], 31533: ['A0A097J0T4', 'O21950'], 69612: ['A0A097J3W3', 'O21953'], 2592190: ['A0A514U5E6'], 2797408: ['A0A7T7K8L4'], 1720494: ['A0A0M7QBP4'], 1204522: ['I7B6T6'], 2218499: ['A0A3G9GR51'], 2591058: ['A0A5B9MVD9'], 2716727: ['A0A6G8R7Q8'], 2716729: ['A0A6G8R8G4'], 2591061: ['A0A5B9MWB5'], 2589644: ['A0A4Y6E901'], 2024309: ['A0A2K9VLC4'], 2686439: ['A0A6B9LI

In [315]:
matches[0]


{'sequence': 'MSDLKSRLIKASTSKLTAELTASKFFNEKDVVRTKIPMMNIALSGEITGGMQSGLLILAGPSKSFKSNFGLTMVSSYMRQYPDAVCLFYDSEFGITPAYLRSMGVDPERVIHTPVQSLEQLRIDMVNQLDAIERGEKVVVFIDSLGNLASKKETEDALNEKVVSDMTRAKTMKSLFRIVTPYFSTKNIPCIAINHTYETQEMFSKTVMGGGTGPMYSADTVFIIGKRQIKDGSDLQGYQFVLNVEKSRTVKEKSKFFIDVKFDGGIDPYSGLLDMALELGFVVKPKNGWYAREFLDEETGEMIREEKSWRAKDTNCTTFWGPLFKHQPFRDAIKRAYQLGAIDSNEIVEAEVDELINSKVEKFKSPESKSKSAADLETDLEQLSDMEEFNE',
 'files': defaultdict(list,
             {'uniprot_query_search.fasta': ['P04529',
               'A0A7S9XGY1',
               'A0A097J734',
               'A0A097J7Q4',
               'A0A023ZVM8',
               'A0AB39C9X7'],
              'ncbi_query_search.fasta': ['P04529',
               'WZX06288',
               'NP_049656',
               'UJD20257',
               'UJD19990',
               'QPI17267',
               'AAD42669',
               'AHY83907',
               'AHY83717',
               'AHY83516',
               'AIT75226',
               'AIT74954',
             

In [ ]:
####
# Get all unique accessions
# For each accession what is the sequence
# Does that sequence match any other sequence in the fastas
# If not, that accession is fully unique
# If it does, are they from different species.
# If different species, keep both as unique
# if no this is redundancy and should be flagged

In [7]:
# Create a function that takes an accession and database file, and returns the sequence
def ExtractSequence(accession, filename, header=False):
    seq = ""
    capture = False

    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()

            if line.startswith(">"):
                if header:
                    header=line
                # check if this is the header we want
                if accession in line:
                    capture = True
                    continue
                else:
                    # stop if we hit a new header after capturing
                    if capture:
                        break
                    capture = False

            elif capture:
                seq += line

    if header:
        return header, seq
    return seq

In [ ]:
ncbi_accessions=[]
csv_path = "../uvsx/data/curated_database/ncbi_query_search_metadata.csv"
with open(csv_path, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        accession = row.get("accession")
        if accession:
            ncbi_accessions.append(accession)
print(f"NCBI Accessions: {len(ncbi_accessions)}")

#Get all query search accessions
uniprot_search_accessions=[]
with open("../uvsx/data/curated_database/uniprot_query_search_metadata.json") as f:
    data = json.load(f)
    for entry in data:
        accession = entry.get("primaryAccession")  # UniProt accession
        uniprot_search_accessions.append(accession)
print(f"Uniprot Accessions: {len(uniprot_search_accessions)}")

#Get all interpro match accessions
domain_match_accessions=[]
with open("../uvsx/data/curated_database/interpro_domain_matches_metadata.json") as f:
    data = json.load(f)
    for entry in data:
        accession = entry.get("primaryAccession")  # UniProt accession
        domain_match_accessions.append(accession)
print(f"Domain match Accessions: {len(domain_match_accessions)}")


unique_accessions=[]
for acc in uniprot_search_accessions:
    if acc not in unique_accessions:
        unique_accessions.append(acc)
for acc in ncbi_accessions:
    if acc not in unique_accessions:
        unique_accessions.append(acc)
for acc in domain_match_accessions:
    if acc not in unique_accessions:
        unique_accessions.append(acc)

In [8]:
# Full script

from collections import defaultdict

files = [
    '../uvsx/data/curated_database/uniprot_query_search.fasta',
    '../uvsx/data/curated_database/ncbi_query_search.fasta',
    '../uvsx/data/curated_database/interpro_domain_matches.fasta'
]

seq_map = {}

for accession in unique_accessions:
    seq = ''
    for file in files:
        seq = ExtractSequence(accession, file)
        if seq:
            break
    seq_map[accession] = seq



sequence_groups = defaultdict(list)

for acc, seq in seq_map.items():
    if seq:  # ignore empty
        sequence_groups[seq].append(acc)

duplicates = {
    seq: accs for seq, accs in sequence_groups.items() if len(accs) > 1
}

import json
import pandas as pd
import pandas as pd
import json

# NCBI CSV
ncbi_df = pd.read_csv('../uvsx/data/curated_database/ncbi_query_search_metadata.csv', low_memory=False)

# UniProt JSON
with open('../uvsx/data/curated_database/uniprot_query_search_metadata.json') as f:
    uniprot_data = json.load(f)

# InterPro JSON
with open('../uvsx/data/curated_database/interpro_domain_matches_metadata.json') as f:
    interpro_data = json.load(f)

ncbi_lookup = dict(zip(ncbi_df["accession"], ncbi_df["taxonId"]))

uniprot_lookup = {
    entry.get("primaryAccession"): entry.get("organism", {}).get("taxonId")
    for entry in uniprot_data
}

interpro_lookup = {
    entry.get("primaryAccession"): entry.get("organism", {}).get("taxonId")
    for entry in interpro_data
}

non_dupe_accessions=[]
for group in sequence_groups.values():

    tax_results = {}

    for acc in group:
        tax = (
            uniprot_lookup.get(acc)
            or ncbi_lookup.get(acc)
            or interpro_lookup.get(acc)
        )

        tax_results[acc] = tax

    non_dupe_accessions.append(acc)


print(len(non_dupe_accessions))

#Adding p04529 to non dupes. Replacing the accession being used instead

i = non_dupe_accessions.index('ADJ39758')
non_dupe_accessions[i] = 'P04529'




15508


True

In [9]:
with open('../uvsx/data/curated_database/cleaned_curated_database.fasta', 'w') as f:
    for acc in non_dupe_accessions:
        header, seq=ExtractSequence(acc, '../uvsx/data/curated_database/ncbi_query_search.fasta', header=True)
        if seq=='':
            seq=ExtractSequence(acc, '../uvsx/data/curated_database/uniprot_query_search.fasta')
        f.write(f'{header} \n')
        for i in range(0, len(seq), 60):
            f.write(seq[i:i+60] + "\n")

[{'entryType': 'UniProtKB reviewed (Swiss-Prot)',
  'primaryAccession': 'P04529',
  'secondaryAccessions': ['Q9MBK8'],
  'uniProtkbId': 'UVSX_BPT4',
  'entryAudit': {'firstPublicDate': '1987-08-13',
   'lastAnnotationUpdateDate': '2026-01-28',
   'lastSequenceUpdateDate': '2001-07-11',
   'entryVersion': 119,
   'sequenceVersion': 2},
  'annotationScore': 3.0,
  'organism': {'scientificName': 'Enterobacteria phage T4',
   'commonName': 'Bacteriophage T4',
   'taxonId': 10665,
   'lineage': ['Viruses',
    'Duplodnaviria',
    'Heunggongvirae',
    'Uroviricota',
    'Caudoviricetes',
    'Pantevenvirales',
    'Straboviridae',
    'Tevenvirinae',
    'Tequatrovirus']},
  'organismHosts': [{'scientificName': 'Escherichia coli', 'taxonId': 562}],
  'proteinExistence': '1: Evidence at protein level',
  'proteinDescription': {'recommendedName': {'fullName': {'value': 'Recombination and repair protein'}}},
  'genes': [{'geneName': {'value': 'UVSX'}}],
  'comments': [{'texts': [{'value': 'Im